# Youtube Transcipts Q&A

In [1]:
import pickle
from pydantic import BaseModel
from openai import OpenAI
from minsearch import Index
import json

### Data Pipeline

In [2]:
!wget -q https://github.com/alexeygrigorev/ai-bootcamp-codespace/raw/refs/heads/main/week1/ph1PxZIkz1o.bin

In [3]:
video_id = 'ph1PxZIkz1o'

with open(f'{video_id}.bin', 'rb') as f_in:
    transcript = pickle.load(f_in)

In [4]:
def format_timestamp(seconds: float) -> str:
    """Convert seconds to H:MM:SS if > 1 hour, else M:SS"""
    total_seconds = int(seconds)
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)

    if hours > 0:
        return f"{hours}:{minutes:02}:{secs:02}"
    else:
        return f"{minutes}:{secs:02}"

def make_subtitles(transcript) -> str:
    lines = []

    for entry in transcript:
        ts = format_timestamp(entry.start)
        text = entry.text.replace('\n', ' ')
        lines.append(ts + ' ' + text)

    return '\n'.join(lines)

subtitles = make_subtitles(transcript)

In [5]:
print(subtitles[:100])

0:00 So hi everyone. Uh today we are going to
0:02 talk about our upcoming course. The
0:05 upcoming


In [5]:
print(transcript[:5])

[FetchedTranscriptSnippet(text='So hi everyone. Uh today we are going to', start=0.0, duration=5.04), FetchedTranscriptSnippet(text='talk about our upcoming course. The', start=2.96, duration=3.52), FetchedTranscriptSnippet(text='upcoming course is called machine', start=5.04, duration=5.92), FetchedTranscriptSnippet(text='learning zoom camp. And um this is', start=6.48, duration=5.92), FetchedTranscriptSnippet(text='already I put the link in the', start=10.96, duration=3.599)]


### Structured RAG output

In [6]:
# define structured classes
class Chapter(BaseModel):
    timestamp: str
    title: str

class YTSummaryResponse(BaseModel):
    summary: str
    chapters: list[Chapter]

In [7]:
openai_client = OpenAI()

In [8]:
instructions = """
    Summarize the transcript and describe the main purpose of the video
    and the main ideas. 

    Also output chapters with time. Use usual sentence case, not Title Case for the chapter.

    More chapters is better than fewer chapters. Have a chapter at least every 3-5 minutes
""".strip()

messages = [
    {"role": "system", "content": instructions}, 
    {"role": "user", "content": subtitles}
]

response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    text_format=YTSummaryResponse # specify the output format
)

In [9]:
summary = response.output_parsed

print(summary.summary)
print()
for c in summary.chapters:
    print(c.timestamp, c.title)

The video introduces the upcoming "Machine Learning Zoom Camp," which starts on September 15. The speaker outlines the structure of the course, modules to expect, and the kind of preparation needed for participants. He emphasizes that it's aimed at aspiring machine learning engineers, focusing more on engineering aspects like deployment rather than deep theoretical knowledge. Participants are encouraged to ask questions live, and several common queries regarding prerequisites, job opportunities, and the course content are addressed. The speaker underlines the hands-on approach of the course, noting that even with limited prior knowledge, motivated learners can benefit greatly. Additionally, there are opportunities for portfolio projects and certificates upon completion.

0:00 Introduction to the course
5:00 Job placement and skills acquired
10:00 Prerequisites for the course
15:00 Deep learning and computer vision coverage
20:00 Course materials and companion book
25:00 Project expecta

### Chunking

In [10]:
def sliding_window(seq, size, step):
    """Create overlapping chunks using sliding window approach."""
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i:i+size] # get the next sequence of <size> characters
        result.append(batch)
        if i + size >= n:
            break

    return result

In [11]:
def join_lines(transcript) -> str:
    """Join transcript entries into continuous text."""
    lines = []

    for entry in transcript:
        text = entry.text.replace('\n', ' ')
        lines.append(text)

    return ' '.join(lines)

In [12]:
def format_chunk(chunk):
    """Format a chunk with start/end timestamps and text."""
    time_start = format_timestamp(chunk[0].start) # start timestamp of the first snippet in the chunk
    time_end = format_timestamp(chunk[-1].start) # start timestamp of the last snippet in the chunk
    text = join_lines(chunk) # string together all the snipets' text content

    return {
        'start': time_start,
        'end': time_end,
        'text': text
    }

In [13]:
chunks = []

for chunk in sliding_window(transcript, 60, 30):
    processed = format_chunk(chunk)
    chunks.append(processed)

print(f"Created {len(chunks)} chunks")

Created 46 chunks


### RAG

In [14]:
index = Index(text_fields=["text"])
index.fit(chunks)

In [15]:
def search(query):
    """Search for the 15 most relevant chunks."""
    return index.search(
        query=query,
        num_results=15
    )

In [16]:
prompt_template = """
    <VIDEO_ID>
    {video_id}
    </VIDEO_ID>

    <QUESTION>
    {question}
    </QUESTION>

    <CONTEXT>
    {context}
    </CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results)
    return prompt_template.format(
        question=question,
        context=context,
        video_id=video_id
    ).strip()

In [17]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [18]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    response = llm(prompt, instructions=instructions)
    return response

In [19]:
answer = rag('Can I find a job after the course?')
print(answer)

### Summary

The video discusses the upcoming "Machine Learning Zoom Camp," providing information about the course structure, contents, and answering participant questions about job readiness after completing the course. The instructor emphasizes that while the course provides essential skills, job placement is not provided by the program itself. However, past participants have successfully found jobs after completion, indicating a good chance of employment. The video also touches on the importance of project-based learning and encourages participants to engage in hands-on projects, volunteer opportunities, and building a portfolio to enhance job readiness.

### Main Purpose
The main purpose of the video is to introduce and provide detailed information about the "Machine Learning Zoom Camp," showcasing its curriculum, resources, and potential career outcomes for participants.

### Main Ideas
- **Course Structure**: The course consists of pre-recorded video lectures and a mixture of pro